# KLOD — primeiro ciclo de treinamento QLoRA

Este notebook executa continuação de pré-treinamento do Klod 1.7B em GPU NVIDIA do Google Colab. Ele **não** cria um modelo do zero: ajusta um adaptador LoRA sobre Qwen3-1.7B-Base. Comece com um ciclo curto e avalie antes de aumentar os passos.

## 1. Ative uma GPU
No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** (ou GPU NVIDIA com pelo menos 12 GiB). Não continue se `nvidia-smi` não mostrar uma GPU.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA ausente. Ative um runtime GPU antes de continuar.'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')

## 2. Baixe o código da branch development e instale as dependências

In [ ]:
%cd /content
!rm -rf KLOD-AI
!git clone --depth 1 -b development https://github.com/japaguimarassi-create/Security-lab.git KLOD-AI
%cd /content/KLOD-AI
!pip install -e '.[data,training]'

## 3. Importe uma amostra de português e prepare os dados
FineWeb2 é uma fonte de pré-treinamento. O importador limita a quantidade de documentos e preserva URL, licença e atribuição. Este primeiro notebook usa apenas português para validar o ciclo; a mistura multilíngue completa está em `configs/data_mixture.yaml`.

In [ ]:
!python scripts/import_fineweb2.py --language por_Latn --limit 50000 --output data/raw/pt.jsonl
!python scripts/prepare_data.py --input data/raw/pt.jsonl --output data/processed/pt.jsonl --min-chars 500

## 4. Crie uma mistura experimental e reduza o primeiro ciclo
O ajuste de 250 passos é apenas um primeiro experimento. Não o trate como treinamento completo de uma LLM. O resultado é um adaptador pequeno, não os pesos-base de 1.7B.

In [ ]:
from pathlib import Path
import yaml
mix = {'sources': [{'name': 'portuguese', 'path': 'data/processed/pt.jsonl', 'weight': 1.0}]}
Path('configs/data_mixture_colab.yaml').write_text(yaml.safe_dump(mix, sort_keys=False), encoding='utf-8')
!python scripts/build_mixture.py --config configs/data_mixture_colab.yaml --output data/processed/mixture.jsonl --max-documents 50000
cfg_path = Path('configs/training/continued_pretraining.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
cfg.update({'max_steps': 250, 'max_sequence_length': 1024, 'eval_steps': 50, 'save_steps': 50, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8, 'load_in_4bit': True})
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
!python scripts/training_preflight.py --stage pretrain

## 5. Execute o primeiro ciclo
A execução exige GPU CUDA e `bitsandbytes`. Se houver erro de memória, reduza `max_sequence_length` para 512 e mantenha batch size 1. Não aumente os passos até revisar a perda de validação.

In [ ]:
!python scripts/train_pretrain.py --config configs/training/continued_pretraining.yaml

## 6. Salve o adaptador fora do GitHub
O checkpoint contém apenas o adaptador LoRA. Guarde-o em armazenamento persistente e anote o commit do código, a versão do modelo-base e os dados usados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/KLOD/checkpoints
!cp -r outputs/klod-continued-pretraining /content/drive/MyDrive/KLOD/checkpoints/
print('Adaptador salvo no Google Drive.')

## O que vem depois
1. Produza um conjunto SFT revisado com pelo menos 20 conversas no formato `{"messages":[{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}`.
2. Configure `previous_adapter` em `configs/training/sft.yaml` para o adaptador salvo e execute `scripts/train_sft.py`.
3. Produza pelo menos 100 pares DPO revisados (`prompt`, `chosen`, `rejected`) e execute `scripts/train_dpo.py`.
4. Avalie em um conjunto separado que nunca entrou no treino.

Não publique dados privados, tokens, chaves ou checkpoints grandes em commits públicos.